In [ ]:
import pandas as pd
from pathlib import Path

# load raw data
full_df = pd.read_csv("../data/raw_data/full_dataset.csv", sep='\t')
cat_map = pd.read_csv("../data/raw_data/category_mapping.csv", sep='\t')

print(f"full_df: {full_df.shape}, {full_df.columns.tolist()}")
print(f"cat_map: {cat_map.shape}, {cat_map.columns.tolist()}")

full_df: (502310, 4), ['offer_id', 'text', 'clean_category_id', 'noisy_category_id']
cat_map: (5692, 2), ['category_label', 'category_name']


In [ ]:
# split category path into separate level columns
levels = cat_map['category_name'].str.split(' > ', expand=True)
levels.columns = [f'L{i+1}' for i in range(levels.shape[1])]
cat_map = pd.concat([cat_map, levels], axis=1)

print(f"Максимальная глубина: {levels.shape[1]}")  # depth goes from 3 to 9
print(cat_map.iloc[0][['category_name'] + [f'L{i}' for i in range(1,6)]])

# merge category info into main dataframe
# once for clean labels, once for noisy
cat_indexed = cat_map.set_index('category_label')

full_df = full_df.merge(
    cat_indexed.add_suffix('_clean'),
    left_on='clean_category_id',
    right_index=True,
    how='left'
)

full_df = full_df.merge(
    cat_indexed.add_suffix('_noisy'),
    left_on='noisy_category_id',
    right_index=True,
    how='left'
)

# save to parquet so we dont have to redo this every time
PROC_DIR = Path("../data/prepared_data")
PROC_DIR.mkdir(exist_ok=True)

full_df.to_parquet(PROC_DIR / "processed_dataset.parquet", index=False)
display(full_df[['text', 'clean_category_id', 'L1_clean', 'L2_clean', 'L3_clean', 'L4_clean', 'L1_noisy', 'L2_noisy']].head(3))

Максимальная глубина: 9
category_name    Allegro > Culture and Entertainment > Movies >...
L1                                                         Allegro
L2                                       Culture and Entertainment
L3                                                          Movies
L4                                                            DVDs
L5                                                           Anime
Name: 0, dtype: object


,text,clean_category_id,L1_clean,L2_clean,L3_clean,L4_clean,L1_noisy,L2_noisy
0,VU meter LED stereo audio control indicator,19,Allegro,Electronics,"Audio, Video and Household Appliances",Home Audio,Allegro,Electronics
1,UV meter stereo audio control indicators,19,Allegro,Electronics,"Audio, Video and Household Appliances",Home Audio,Allegro,Electronics
2,Taga Harmony PF-1000 v.2 black - power lines,19,Allegro,Electronics,"Audio, Video and Household Appliances",Home Audio,Allegro,Electronics
